# Auto-Prompt Case Study Analysis

This notebook analyzes the results of prompt optimization experiments.

In [ ]:
import json
import pandas as pd
from pathlib import Path
import sys

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import visualization utilities
try:
    from shared.visualization import plot_field_comparison, plot_accuracy_bars, plot_improvement_heatmap, save_figure
    PLOTS_AVAILABLE = True
except ImportError:
    print("Visualization utilities not available. Install matplotlib.")
    PLOTS_AVAILABLE = False

## Load Results

In [ ]:
# Load experiment results
RESULTS_DIR = PROJECT_ROOT / "experiments" / "resume_extraction" / "results"

def load_results(results_dir):
    results = {}
    for filename in ["baseline_results.json", "dspy_results.json", "comparison_results.json"]:
        filepath = results_dir / filename
        if filepath.exists():
            with open(filepath) as f:
                results[filename.replace('.json', '')] = json.load(f)
    return results

results = load_results(RESULTS_DIR)
print(f"Loaded results: {list(results.keys())}")

## Summary Statistics

In [ ]:
if 'baseline_results' in results and 'dspy_results' in results:
    baseline = results['baseline_results']
    dspy = results['dspy_results']
    
    print("=" * 50)
    print("OVERALL ACCURACY")
    print("=" * 50)
    print(f"Baseline: {baseline['overall_accuracy']:.2%}")
    print(f"DSPy:     {dspy['overall_accuracy']:.2%}")
    print(f"Improvement: {dspy['overall_accuracy'] - baseline['overall_accuracy']:.2%}")
    
    print("\n" + "=" * 50)
    print("FIELD-WISE ACCURACY")
    print("=" * 50)
    for field in baseline['field_accuracies']:
        b_acc = baseline['field_accuracies'][field]
        d_acc = dspy['field_accuracies'][field]
        print(f"{field:20} Baseline: {b_acc:6.2%}  DSPy: {d_acc:6.2%}  ({d_acc - b_acc:+.2%})")
else:
    print("Results not found. Run the experiment first.")

## Visualizations

In [ ]:
if PLOTS_AVAILABLE and 'baseline_results' in results:
    # Field comparison
    field_data = {
        'Baseline': results['baseline_results']['field_accuracies'],
        'DSPy': results['dspy_results']['field_accuracies']
    }
    fig = plot_field_comparison(field_data, title="Field-wise Accuracy: Baseline vs DSPy")
    save_figure(fig, Path("figures/field_comparison"))
    fig.show()

In [ ]:
if PLOTS_AVAILABLE and 'baseline_results' in results:
    # Overall accuracy bars
    overall_data = {
        'Baseline': results['baseline_results']['overall_accuracy'],
        'DSPy': results['dspy_results']['overall_accuracy']
    }
    fig = plot_accuracy_bars(overall_data, title="Overall Accuracy Comparison")
    save_figure(fig, Path("figures/overall_accuracy"))
    fig.show()

In [ ]:
if PLOTS_AVAILABLE and 'comparison_results' in results:
    # Improvement heatmap
    fig = plot_improvement_heatmap(results['comparison_results'], title="Per-Sample Improvement Analysis")
    save_figure(fig, Path("figures/improvement_heatmap"))
    fig.show()

## Error Analysis

In [ ]:
if 'comparison_results' in results:
    comparison = results['comparison_results']
    
    print(f"Samples improved: {comparison['summary']['total_improvements']}")
    print(f"Samples degraded: {comparison['summary']['total_degradations']}")
    print(f"Samples unchanged: {comparison['summary']['total_unchanged']}")
    
    if comparison['improvements']:
        print("\nTop 5 Improvements:")
        for imp in sorted(comparison['improvements'], key=lambda x: x['difference'], reverse=True)[:5]:
            print(f"  Sample {imp['sample_id']}: {imp['baseline_score']:.2f} -> {imp['optimized_score']:.2f} (+{imp['difference']:.2f})")

## Export Summary Table

In [ ]:
if 'baseline_results' in results:
    # Create summary dataframe
    summary_data = []
    for field in results['baseline_results']['field_accuracies']:
        summary_data.append({
            'Field': field,
            'Baseline': f"{results['baseline_results']['field_accuracies'][field]:.2%}",
            'DSPy': f"{results['dspy_results']['field_accuracies'][field]:.2%}",
            'Improvement': f"{results['dspy_results']['field_accuracies'][field] - results['baseline_results']['field_accuracies'][field]:+.2%}"
        })
    
    # Add overall row
    summary_data.append({
        'Field': 'OVERALL',
        'Baseline': f"{results['baseline_results']['overall_accuracy']:.2%}",
        'DSPy': f"{results['dspy_results']['overall_accuracy']:.2%}",
        'Improvement': f"{results['dspy_results']['overall_accuracy'] - results['baseline_results']['overall_accuracy']:+.2%}"
    })
    
    df = pd.DataFrame(summary_data)
    print(df.to_markdown(index=False))